# Silver Layer

In [0]:
from pyspark.sql.functions import col, to_timestamp, hour, minute, date_format


df_trips = spark.table("demo.demo.bronze_taxi_trips")
df_zones = spark.table("demo.demo.bronze_taxi_zones")

In [0]:
df_trips = df_trips.withColumn("pickup_datetime", to_timestamp(col("tpep_pickup_datetime"), "dd/MM/yy H:mm")).withColumn("dropoff_datetime", to_timestamp(col("tpep_dropoff_datetime"), "dd/MM/yy H:mm"))



In [0]:
#clean data
df_trips = df_trips.filter(
    col("pickup_datetime").isNotNull() & 
    col("dropoff_datetime").isNotNull() &
    col("PULocationID").isNotNull() &
    col("DOLocationID").isNotNull() &
    (col("trip_distance") > 0) &
    (col("total_amount") > 0)
)

In [0]:
#extract only time features
df_drips_clean = df_trips\
    .withColumn("pickup_hour", hour(col("pickup_datetime")))\
    .withColumn("pickup_minute", minute(col("pickup_datetime")))\
    .withColumn("dropoff_hour", hour(col("dropoff_datetime")))\
    .withColumn("pickup_time_only", date_format(col("pickup_datetime"), "HH:mm:ss"))\
    .withColumn("dropoff_time_only", date_format(col("dropoff_datetime"), "HH:mm:ss"))


In [0]:
from pyspark.sql.functions import col

df_enrich = df_drips_clean.alias("trips")\
    .join(df_zones.alias("pu"),
              col("trips.PULocationID") == col("pu.LocationID"), "left")\
    .join(df_zones.alias("do"),
              col("trips.DOLocationID") == col("do.LocationID"), "left")

In [0]:
df_final = df_enrich.select(
    col("trips.*"),
        
    col("pu.ZoneName").alias("pickup_zone"),
    col("pu.Latitude").alias("pickup_latitude"),
    col("pu.Longitude").alias("pickup_longitude"),
    col("do.ZoneName").alias("dropoff_zone"),
    col("do.Latitude").alias("dropoff_latitude"),
    col("do.Longitude").alias("dropoff_longitude")
)

df_final.write.format("delta")\
    .mode("overwrite") \
    .option("mergeSchema", "true")\
    .saveAsTable("demo.demo.silver_taxi_trips_enriched")

